In [5]:
!py -m pip install torch transformers --upgrade

In [15]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline
from langchain.chains import RetrievalQA


In [2]:
# load pdf and split into chunks
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# load the pdf file
loader = PyPDFLoader("data/medical.pdf")
documents = loader.load()

print(f"Total pages in PDF: {len(documents)}")

# split text into smaller chunks for better retrieval
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = text_splitter.split_documents(documents)


Total pages in PDF: 599


In [1]:
# RAG pipeline: pdf to faiss vector store
!py -m pip install pypdf sentence-transformers faiss-cpu -q

import warnings
warnings.filterwarnings('ignore')
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

 #Step 1: Load PDF and extract text
reader = PdfReader("data/medical.pdf")
text = "".join([page.extract_text() for page in reader.pages])

#Step 2: Split text into chunks of 500 words
words = text.split()
chunks = [" ".join(words[i:i+500]) for i in range(0, len(words), 500)]

#Step 3: Create embeddings using SentenceTransformer
embedder = SentenceTransformer('all-MiniLM-L6-v2')
chunk_embeddings = embedder.encode(chunks, show_progress_bar=True)

#Step 4: Build FAISS index for vector search
index = faiss.IndexFlatL2(chunk_embeddings.shape[1])
index.add(chunk_embeddings.astype('float32'))
# Step 5: Function to retrieve relevant context
def get_context(question, k=2):
    question_emb = embedder.encode([question])
    D, I = index.search(question_emb.astype('float32'), k)
    relevant_chunks = [chunks[i] for i in I[0]]
    return "\n\n".join(relevant_chunks)

# Test the context retrieval function
context = get_context("What are the symptoms of diabetes?")
context = get_context("What is the treatment for high blood pressure?")

# Load FLAN-T5 model for Question Answering
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

def get_answer(question):
    question_emb = embedder.encode([question])
    D, I = index.search(question_emb.astype('float32'), k=1)
    context = chunks[I[0][0]][:300]
    input_text = f"Answer in 2-3 sentences only. question:{question} context: {context}"
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids
    outputs = model.generate(input_ids, max_length=60)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer

# Test the final function
question = "What are the symptoms of diabetes?"
answer = get_answer(question)

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 2107.67it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
